# Model Training & Evaluation
Training and evaluating all models in the Healthcare Intelligence System.

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# PySpark imports
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# ML imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc, 
                             roc_auc_score, precision_recall_fscore_support)
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.model_selection import train_test_split

# Project imports
from src.data_processing.spark_session import create_spark_session
from src.data_processing.data_loader import HealthcareDataLoader
from src.feature_engineering.structured_features import StructuredFeatureEngineer
from src.feature_engineering.nlp_features import NLPFeatureEngineer
from src.feature_engineering.symptom_features import SymptomFeatureEngineer
from src.feature_engineering.lab_features import LabFeatureEngineer
from src.models.symptom_classifier import SymptomClassifier
from src.models.clinical_nlp_model import ClinicalNLPModel
from src.models.lab_anomaly_detector import LabAnomalyDetector
from src.models.image_classifier import MedicalImageClassifier
from src.models.ensemble_model import EnsembleModel

plt.style.use('seaborn-v0_8-whitegrid')
print("All imports successful!")

## Data Preparation

In [ ]:
# Create Spark session and load all data
spark = create_spark_session(app_name="ModelTraining", master="local[*]")
loader = HealthcareDataLoader(spark, data_dir=os.path.join(project_root, 'data'))

patients_df = loader.load_patients()
vitals_df = loader.load_vitals()
symptoms_df = loader.load_symptoms()
lab_results_df = loader.load_lab_results()
clinical_notes_df = loader.load_clinical_notes()
ground_truth_df = loader.load_ground_truth()

# Engineer features from all modalities
struct_eng = StructuredFeatureEngineer(spark)
symptom_eng = SymptomFeatureEngineer(spark)
lab_eng = LabFeatureEngineer(spark)
nlp_eng = NLPFeatureEngineer(spark)

structured_features = struct_eng.engineer_features(patients_df, vitals_df)
symptom_features = symptom_eng.engineer_features(symptoms_df)
lab_features = lab_eng.engineer_features(lab_results_df)
nlp_features = nlp_eng.engineer_features(clinical_notes_df)

# Convert to pandas for sklearn-based models
ground_truth_pd = ground_truth_df.toPandas()
le = LabelEncoder()
ground_truth_pd['label'] = le.fit_transform(ground_truth_pd['diagnosis'])
class_names = le.classes_

# Train/test split (80/20 stratified)
train_ids, test_ids = train_test_split(
    ground_truth_pd['patient_id'], 
    test_size=0.2, 
    random_state=42, 
    stratify=ground_truth_pd['diagnosis']
)

print(f"Train set: {len(train_ids)} patients")
print(f"Test set: {len(test_ids)} patients")
print(f"Classes: {list(class_names)}")
print(f"\nClass distribution (train):")
train_gt = ground_truth_pd[ground_truth_pd['patient_id'].isin(train_ids)]
print(train_gt['diagnosis'].value_counts().to_string())

# Store results for comparison
model_results = {}

## Model 1: Symptom Classifier (PySpark MLlib Random Forest)

In [ ]:
# Train Symptom Classifier with CrossValidator
symptom_clf = SymptomClassifier(spark)

# Prepare training and test data
train_ids_list = train_ids.tolist()
test_ids_list = test_ids.tolist()

# Train the model with cross-validation
symptom_model, symptom_metrics = symptom_clf.train(
    symptom_features, ground_truth_df, 
    train_patient_ids=train_ids_list,
    test_patient_ids=test_ids_list
)

# Print results
print("=" * 60)
print("SYMPTOM CLASSIFIER - Results")
print("=" * 60)
print(f"\nBest hyperparameters:")
if hasattr(symptom_model, 'bestModel'):
    best = symptom_model.bestModel
    print(f"  numTrees: {best.getNumTrees if hasattr(best, 'getNumTrees') else 'N/A'}")
    print(f"  maxDepth: {best.getOrDefault('maxDepth') if hasattr(best, 'getOrDefault') else 'N/A'}")

print(f"\nTest Metrics:")
for metric_name, metric_val in symptom_metrics.items():
    print(f"  {metric_name}: {metric_val:.4f}")

model_results['Symptom Classifier'] = symptom_metrics

In [ ]:
# Feature importance and confusion matrix for Symptom Classifier
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Feature importance (top 20)
predictions_df = symptom_clf.predict(symptom_features, test_patient_ids=test_ids_list)
feature_importance = symptom_clf.get_feature_importance()

if feature_importance is not None and len(feature_importance) > 0:
    fi_df = pd.DataFrame(feature_importance).sort_values('importance', ascending=False).head(20)
    axes[0].barh(range(len(fi_df)), fi_df['importance'].values, color='forestgreen', edgecolor='black')
    axes[0].set_yticks(range(len(fi_df)))
    axes[0].set_yticklabels(fi_df['feature'].values)
    axes[0].invert_yaxis()
    axes[0].set_title('Top 20 Feature Importance - Symptom Classifier', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Importance')
else:
    axes[0].text(0.5, 0.5, 'Feature importance\nnot available', transform=axes[0].transAxes,
                 ha='center', va='center', fontsize=14)

# Confusion matrix
if predictions_df is not None:
    pred_pd = predictions_df.toPandas() if hasattr(predictions_df, 'toPandas') else predictions_df
    y_true_col = [c for c in pred_pd.columns if 'label' in c.lower() or 'true' in c.lower() or 'diagnosis' in c.lower()][0]
    y_pred_col = [c for c in pred_pd.columns if 'prediction' in c.lower() or 'pred' in c.lower()][0]
    
    cm = confusion_matrix(pred_pd[y_true_col], pred_pd[y_pred_col])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
                xticklabels=class_names, yticklabels=class_names)
    axes[1].set_title('Confusion Matrix - Symptom Classifier', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig(os.path.join(project_root, 'outputs', 'symptom_classifier_results.png'), dpi=150, bbox_inches='tight')
plt.show()

## Model 2: Clinical NLP Model

In [ ]:
# Train Clinical NLP Model
nlp_model = ClinicalNLPModel(spark)
nlp_trained, nlp_metrics = nlp_model.train(
    nlp_features, ground_truth_df,
    train_patient_ids=train_ids_list,
    test_patient_ids=test_ids_list
)

print("=" * 60)
print("CLINICAL NLP MODEL - Results")
print("=" * 60)
print(f"\nTest Metrics:")
for metric_name, metric_val in nlp_metrics.items():
    print(f"  {metric_name}: {metric_val:.4f}")

# Per-class metrics
nlp_predictions = nlp_model.predict(nlp_features, test_patient_ids=test_ids_list)
if nlp_predictions is not None:
    nlp_pred_pd = nlp_predictions.toPandas() if hasattr(nlp_predictions, 'toPandas') else nlp_predictions
    y_true_col = [c for c in nlp_pred_pd.columns if 'label' in c.lower() or 'true' in c.lower() or 'diagnosis' in c.lower()][0]
    y_pred_col = [c for c in nlp_pred_pd.columns if 'prediction' in c.lower() or 'pred' in c.lower()][0]
    
    print(f"\nPer-Class Classification Report:")
    print(classification_report(nlp_pred_pd[y_true_col], nlp_pred_pd[y_pred_col], target_names=class_names))

model_results['Clinical NLP'] = nlp_metrics

## Model 3: Lab Anomaly Detector

In [ ]:
# Train Lab Anomaly Detector (Isolation Forest + KMeans)
lab_detector = LabAnomalyDetector(spark)
lab_model, lab_metrics = lab_detector.train(
    lab_features, ground_truth_df,
    train_patient_ids=train_ids_list,
    test_patient_ids=test_ids_list
)

print("=" * 60)
print("LAB ANOMALY DETECTOR - Results")
print("=" * 60)
print(f"\nTest Metrics:")
for metric_name, metric_val in lab_metrics.items():
    if isinstance(metric_val, float):
        print(f"  {metric_name}: {metric_val:.4f}")
    else:
        print(f"  {metric_name}: {metric_val}")

# Cluster analysis visualization
lab_pred = lab_detector.predict(lab_features, test_patient_ids=test_ids_list)
if lab_pred is not None:
    lab_pred_pd = lab_pred.toPandas() if hasattr(lab_pred, 'toPandas') else lab_pred
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Anomaly score distribution
    if 'anomaly_score' in lab_pred_pd.columns:
        axes[0].hist(lab_pred_pd['anomaly_score'], bins=30, edgecolor='black', alpha=0.7, color='salmon')
        axes[0].set_title('Anomaly Score Distribution', fontsize=13, fontweight='bold')
        axes[0].set_xlabel('Anomaly Score')
        axes[0].set_ylabel('Count')
    
    # Cluster distribution
    cluster_col = [c for c in lab_pred_pd.columns if 'cluster' in c.lower()]
    if cluster_col:
        cluster_counts = lab_pred_pd[cluster_col[0]].value_counts().sort_index()
        axes[1].bar(cluster_counts.index.astype(str), cluster_counts.values, color='steelblue', edgecolor='black')
        axes[1].set_title('Cluster Distribution', fontsize=13, fontweight='bold')
        axes[1].set_xlabel('Cluster')
        axes[1].set_ylabel('Count')
    
    plt.tight_layout()
    plt.savefig(os.path.join(project_root, 'outputs', 'lab_anomaly_results.png'), dpi=150, bbox_inches='tight')
    plt.show()

model_results['Lab Anomaly Detector'] = lab_metrics

## Model 4: Medical Image Classifier

In [ ]:
# Train Medical Image Classifier (DenseNet-121)
image_clf = MedicalImageClassifier(num_classes=len(class_names))

# Set up image data directory
image_dir = os.path.join(project_root, 'data', 'images')

print("=" * 60)
print("MEDICAL IMAGE CLASSIFIER - DenseNet-121")
print("=" * 60)

# Train on available images (or demonstrate architecture)
try:
    image_model, image_metrics = image_clf.train(
        image_dir=image_dir,
        ground_truth=ground_truth_pd,
        train_patient_ids=train_ids_list,
        test_patient_ids=test_ids_list,
        epochs=10,
        batch_size=16
    )
    
    print(f"\nTraining Results:")
    for metric_name, metric_val in image_metrics.items():
        if isinstance(metric_val, float):
            print(f"  {metric_name}: {metric_val:.4f}")
        elif isinstance(metric_val, list):
            print(f"  {metric_name}: [last={metric_val[-1]:.4f}]")
    
    # Plot training curves
    if 'train_loss' in image_metrics and isinstance(image_metrics['train_loss'], list):
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        axes[0].plot(image_metrics['train_loss'], label='Train Loss', color='blue')
        if 'val_loss' in image_metrics:
            axes[0].plot(image_metrics['val_loss'], label='Val Loss', color='red')
        axes[0].set_title('Training Loss', fontsize=13, fontweight='bold')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].legend()
        
        if 'train_acc' in image_metrics:
            axes[1].plot(image_metrics['train_acc'], label='Train Acc', color='blue')
        if 'val_acc' in image_metrics:
            axes[1].plot(image_metrics['val_acc'], label='Val Acc', color='red')
        axes[1].set_title('Training Accuracy', fontsize=13, fontweight='bold')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy')
        axes[1].legend()
        
        plt.tight_layout()
        plt.savefig(os.path.join(project_root, 'outputs', 'image_training_curves.png'), dpi=150, bbox_inches='tight')
        plt.show()
    
    model_results['Image Classifier'] = image_metrics

except Exception as e:
    print(f"\nNote: Image classifier training skipped or failed: {e}")
    print("This is expected if image data is not available or GPU is not configured.")
    print(f"\nModel Architecture:")
    print(image_clf.get_model_summary() if hasattr(image_clf, 'get_model_summary') else "DenseNet-121 with custom classification head")
    model_results['Image Classifier'] = {'accuracy': 0.0, 'note': 'Training skipped'}

## Ensemble Model

In [ ]:
# Train Ensemble (Meta-Learner) combining all models
ensemble = EnsembleModel(spark)

# Train ensemble with all component models
ensemble_model, ensemble_metrics = ensemble.train(
    models={
        'symptom_classifier': symptom_clf,
        'clinical_nlp': nlp_model,
        'lab_anomaly': lab_detector,
    },
    features={
        'symptom_features': symptom_features,
        'nlp_features': nlp_features,
        'lab_features': lab_features,
    },
    ground_truth_df=ground_truth_df,
    train_patient_ids=train_ids_list,
    test_patient_ids=test_ids_list
)

print("=" * 60)
print("ENSEMBLE MODEL - Results")
print("=" * 60)
print(f"\nTest Metrics:")
for metric_name, metric_val in ensemble_metrics.items():
    if isinstance(metric_val, float):
        print(f"  {metric_name}: {metric_val:.4f}")

model_results['Ensemble'] = ensemble_metrics

In [ ]:
# ROC curves - all models + ensemble on same plot
fig, ax = plt.subplots(figsize=(10, 8))

model_colors = {
    'Symptom Classifier': '#e74c3c',
    'Clinical NLP': '#3498db', 
    'Lab Anomaly Detector': '#2ecc71',
    'Image Classifier': '#9b59b6',
    'Ensemble': '#e67e22'
}

# Collect predictions from each model and plot ROC
all_model_preds = {
    'Symptom Classifier': symptom_clf,
    'Clinical NLP': nlp_model,
    'Lab Anomaly Detector': lab_detector,
}

features_map = {
    'Symptom Classifier': symptom_features,
    'Clinical NLP': nlp_features,
    'Lab Anomaly Detector': lab_features,
}

for model_name, model_obj in all_model_preds.items():
    try:
        preds = model_obj.predict(features_map[model_name], test_patient_ids=test_ids_list)
        if preds is not None:
            pred_pd = preds.toPandas() if hasattr(preds, 'toPandas') else preds
            
            # Look for probability columns
            prob_col = [c for c in pred_pd.columns if 'probability' in c.lower() or 'prob' in c.lower()]
            y_true_col = [c for c in pred_pd.columns if 'label' in c.lower() or 'true' in c.lower()][0]
            
            if prob_col:
                y_true_bin = label_binarize(pred_pd[y_true_col], classes=range(len(class_names)))
                if y_true_bin.shape[1] == 1:
                    y_true_bin = np.hstack([1 - y_true_bin, y_true_bin])
                
                # Macro-average ROC
                fpr, tpr, _ = roc_curve(y_true_bin.ravel(), 
                                         np.array(pred_pd[prob_col[0]].tolist()).ravel())
                roc_auc_val = auc(fpr, tpr)
                ax.plot(fpr, tpr, color=model_colors.get(model_name, 'gray'), 
                       linewidth=2, label=f'{model_name} (AUC = {roc_auc_val:.3f})')
    except Exception as e:
        print(f"  Could not plot ROC for {model_name}: {e}")

# Ensemble ROC
try:
    ensemble_preds = ensemble.predict(
        features={'symptom_features': symptom_features, 'nlp_features': nlp_features, 'lab_features': lab_features},
        test_patient_ids=test_ids_list
    )
    if ensemble_preds is not None:
        ens_pd = ensemble_preds.toPandas() if hasattr(ensemble_preds, 'toPandas') else ensemble_preds
        prob_col = [c for c in ens_pd.columns if 'probability' in c.lower() or 'prob' in c.lower()]
        y_true_col = [c for c in ens_pd.columns if 'label' in c.lower() or 'true' in c.lower()][0]
        
        if prob_col:
            y_true_bin = label_binarize(ens_pd[y_true_col], classes=range(len(class_names)))
            if y_true_bin.shape[1] == 1:
                y_true_bin = np.hstack([1 - y_true_bin, y_true_bin])
            fpr, tpr, _ = roc_curve(y_true_bin.ravel(), np.array(ens_pd[prob_col[0]].tolist()).ravel())
            roc_auc_val = auc(fpr, tpr)
            ax.plot(fpr, tpr, color=model_colors['Ensemble'], linewidth=3, linestyle='--',
                   label=f'Ensemble (AUC = {roc_auc_val:.3f})')
except Exception as e:
    print(f"  Could not plot ensemble ROC: {e}")

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Baseline')
ax.set_title('ROC Curves - All Models Comparison', fontsize=16, fontweight='bold')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'outputs', 'roc_curves_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## Model Comparison Summary

In [ ]:
# Create summary comparison table
comparison_rows = []
standard_metrics = ['accuracy', 'f1_score', 'precision', 'recall', 'auc']

for model_name, metrics in model_results.items():
    row = {'Model': model_name}
    for m in standard_metrics:
        # Try different metric name variations
        for key in metrics:
            if m in key.lower():
                val = metrics[key]
                if isinstance(val, float):
                    row[m.replace('_', ' ').title()] = f"{val:.4f}"
                break
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).set_index('Model')

# Display as styled table
print("=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)
print(comparison_df.to_string())

# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))
metric_cols = [c for c in comparison_df.columns if comparison_df[c].dtype == object]
plot_df = comparison_df.copy()
for col in metric_cols:
    plot_df[col] = pd.to_numeric(plot_df[col], errors='coerce')

plot_df = plot_df.dropna(axis=1, how='all')
if len(plot_df.columns) > 0:
    plot_df.plot(kind='bar', ax=ax, width=0.8, edgecolor='black')
    ax.set_title('Model Performance Comparison', fontsize=16, fontweight='bold')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.1)
    ax.legend(title='Metric', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.tick_params(axis='x', rotation=45)
    
    # Add value labels
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=8, rotation=90, padding=3)

plt.tight_layout()
plt.savefig(os.path.join(project_root, 'outputs', 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

# Stop Spark session
spark.stop()
print("\nSpark session stopped. Model training & evaluation complete!")